# Media Analytics – ETL Pipeline
### Traditional vs. Digital Media Consumption Shift

**Target Schema:** Constellation — 3 fact tables, 5 non-time dimensions, 1 time dimension  
**Fact Tables:** `FACT_MEDIA_PERFORMANCE` (Cumulative), `FACT_ENGAGEMENT` (Snapshot), `FACT_SUBSCRIPTION_PRICING` (Snapshot)  
**SCD:** DIM_PLATFORM → SCD2 | DIM_SUBSCRIPTION_PLAN → SCD3 | others → SCD0/SCD1

---
**Notebook Structure**
- Part 1: Load & Profile All Data Sources
- Part 2: Transformations — streaming_service.csv
- Part 3: Transformations — platform_summary.csv
- Part 4: DIM_PLATFORM Initial Load (SCD2)

In [351]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

---
## Part 1 – Load & Profile All Data Sources

| # | File | Rows | Feeds |
|---|---|---|---|
| 1 | `streaming_service.csv` | 777 | FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN |
| 2 | `platform_summary.csv` | 12 | DIM_PLATFORM |
| 3 | `platform_financials_comprehensive.csv` | 10 | FACT_MEDIA_PERFORMANCE |
| 4 | `industry_metrics.csv` | 9 | FACT_MEDIA_PERFORMANCE |
| 5 | `traditional_media_viewership_monthly.csv` | 1,624 | FACT_MEDIA_PERFORMANCE |
| 6 | `user_engagement_monthly.csv` | 4,120 | FACT_ENGAGEMENT |
| 7 | `switch_factor_survey.csv` | 2,025 | FACT_ENGAGEMENT, DIM_SWITCH_REASON |
| 8 | `platform_subscriber_monthly.csv` | 640 | FACT_MEDIA_PERFORMANCE |

### 1.1 – streaming_service.csv
**Source:** Kaggle | **Feeds:** FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN  
**Known issues:** Date stored as `Jul-2011` string format — needs parsing

In [355]:
StreamingService_df = pd.read_csv('Data Sources copy/streaming_service.csv')

print('Shape:', StreamingService_df.shape)
print('Columns:', list(StreamingService_df.columns))
print('\nData types:')
print(StreamingService_df.dtypes)
print('\nNull counts:')
print(StreamingService_df.isnull().sum())
print('\nSample rows:')
display(StreamingService_df.head(5))
print('\nUnique services:', StreamingService_df['service'].nunique())
print('Date range (raw):', StreamingService_df['date'].min(), '→', StreamingService_df['date'].max())

Shape: (777, 3)
Columns: ['service', 'date', 'price']

Data types:
service     object
date        object
price      float64
dtype: object

Null counts:
service    0
date       0
price      0
dtype: int64

Sample rows:


,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99
3,Netflix,Oct-2011,7.99
4,Netflix,Nov-2011,7.99



Unique services: 10
Date range (raw): Apr-2012 → Sep-2023


### 1.2 – platform_summary.csv
**Source:** Streaming Platform Dataset | **Feeds:** DIM_PLATFORM  
**Known issues:** Wide table (36 cols), many NaN — only 5 dim cols needed

In [358]:
PlatformSummary_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_summary.csv')

print('Shape:', PlatformSummary_df.shape)
print('\nDimension columns (selected for DIM_PLATFORM):')
dim_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
print(PlatformSummary_df[dim_cols].to_string())
print('\nNull counts (dim cols only):')
print(PlatformSummary_df[dim_cols].isnull().sum())

Shape: (12, 36)

Dimension columns (selected for DIM_PLATFORM):
              platform           parent_company                                   content_focus               business_model  launch_year
0              Netflix             Netflix Inc.        Original series, films, licensed content           Subscription + Ads         2007
1              Disney+  The Walt Disney Company  Family, Marvel, Star Wars, National Geographic           Subscription + Ads         2019
2                 Hulu  The Walt Disney Company                    TV shows, next-day broadcast           Subscription + Ads         2008
3   Amazon Prime Video          Amazon.com Inc.                 Original series, movies, sports  Prime bundle + Subscription         2006
4           Paramount+         Paramount Global                      Nickelodeon, CBS, Showtime           Subscription + Ads         2021
5              Peacock                  Comcast                            NBCUniversal content           Su

### 1.3 – platform_financials_comprehensive.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** 10 rows only (1 per platform, snapshot); many sparse columns

In [361]:
PlatformFinancials_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_financials_comprehensive.csv')

print('Shape:', PlatformFinancials_df.shape)
print('Platforms:', list(PlatformFinancials_df['platform']))

# Key financial columns relevant to FACT_MEDIA_PERFORMANCE
fin_cols = ['platform', 'reporting_date', 'subscribers_millions',
            'quarterly_revenue_usd_millions', 'quarterly_profit_usd_millions']
print('\nKey columns:')
display(PlatformFinancials_df[fin_cols])
print('\nNull counts (key cols):')
print(PlatformFinancials_df[fin_cols].isnull().sum())

Shape: (10, 32)
Platforms: ['Netflix', 'Roku', 'Disney+', 'Amazon Prime Video', 'Paramount+', 'YouTube TV', 'AMC Networks', 'ITVX', 'DAZN', 'Twitch']

Key columns:


,platform,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions
0,Netflix,2026-01-19,325.0,12050.0,2410.0
1,Roku,2026-02-11,98.0,1395.0,80.5
2,Disney+,2026-02-07,126.0,NaN,NaN
3,Amazon Prime Video,2026-02-06,NaN,NaN,NaN
4,Paramount+,2026-02-06,NaN,NaN,NaN
5,YouTube TV,2026-02-06,NaN,NaN,NaN
6,AMC Networks,2026-02-10,10.4,594.8,NaN
7,ITVX,2026-02-03,NaN,NaN,NaN
8,DAZN,2026-02-06,NaN,NaN,NaN
9,Twitch,2026-02-01,NaN,NaN,NaN



Null counts (key cols):
platform                          0
reporting_date                    0
subscribers_millions              6
quarterly_revenue_usd_millions    7
quarterly_profit_usd_millions     8
dtype: int64


### 1.4 – industry_metrics.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Annual granularity only; industry-level aggregates, not platform-level

In [364]:
IndustryMetrics_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/industry_metrics.csv')

print('Shape:', IndustryMetrics_df.shape)
print('\nAll rows:')
display(IndustryMetrics_df)
print('\nNull counts:')
print(IndustryMetrics_df.isnull().sum())

Shape: (9, 8)

All rows:


,metric_category,metric_name,value_usd_billions,unit,year,source,value_pct,value_usd_millions
0,Global Content Spending,Total Global Content Spend 2025,245.0,USD Billions,2025,Ampere Analysis [citation:7],NaN,NaN
1,Global Content Spending,Total Global Content Spend 2026,255.0,USD Billions,2026,Ampere Analysis [citation:7],NaN,NaN
2,Global Content Spending,Streamer Content Spend 2026,101.0,USD Billions,2026,Ampere Analysis [citation:7],NaN,NaN
3,Global Content Spending,Streamer Share of Global Spend,NaN,Percentage,2026,Ampere Analysis [citation:7],40.0,NaN
4,Sports Rights,Global Streaming Sports Rights Spend 2025,13.2,USD Billions,2025,Ampere Analysis [citation:3],NaN,NaN
5,Sports Rights,Global Streaming Sports Rights Spend 2026,14.2,USD Billions,2026,Ampere Analysis [citation:3],NaN,NaN
6,Sports Rights,Sports Rights YoY Growth,NaN,Percentage,2026,Ampere Analysis [citation:3],7.0,NaN
7,Advertising,Netflix Ad Revenue 2025,NaN,USD Millions,2025,Netflix Earnings [citation:5],NaN,1500.0
8,Advertising,Netflix Projected Ad Revenue 2026,NaN,USD Millions,2026,Netflix Earnings [citation:5],NaN,3000.0



Null counts:
metric_category       0
metric_name           0
value_usd_billions    4
unit                  0
year                  0
source                0
value_pct             7
value_usd_millions    7
dtype: int64


### 1.5 – traditional_media_viewership_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Mixed date formats (`Jan-2010`, `2010/01`, `March 2010`), some `metric_value` stored as strings (`"13.6M"`), inconsistent `media_type` casing, 25 intentional duplicate rows

In [367]:
TraditionalMedia_df = pd.read_csv('Data Sources copy/traditional_media_viewership_monthly.csv')

print('Shape:', TraditionalMedia_df.shape)
print('Columns:', list(TraditionalMedia_df.columns))
print('\nData types:')
print(TraditionalMedia_df.dtypes)
print('\nNull counts:')
print(TraditionalMedia_df.isnull().sum())
print('\nSample rows:')
display(TraditionalMedia_df.head(8))

print('\n--- Data Quality Issues ---')
print('Duplicate rows:', TraditionalMedia_df.duplicated().sum())
print('\nUnique date formats (sample):', TraditionalMedia_df['report_month'].unique()[:10])
print('\nmedia_type unique values:', TraditionalMedia_df['media_type'].unique())
print('\nmetric_value stored as string (sample):')
str_vals = TraditionalMedia_df[TraditionalMedia_df['metric_value'].astype(str).str.contains('M', na=False)]
print(str_vals[['platform_name','report_month','metric_value']].head(5))

Shape: (1624, 7)
Columns: ['report_month', 'platform_name', 'media_type', 'metric_name', 'metric_value', 'market', 'source']

Data types:
report_month     object
platform_name    object
media_type       object
metric_name      object
metric_value     object
market           object
source           object
dtype: object

Null counts:
report_month     0
platform_name    0
media_type       0
metric_name      0
metric_value     0
market           0
source           0
dtype: int64

Sample rows:


,report_month,platform_name,media_type,metric_name,metric_value,market,source
0,Jan-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.63,US,Nielsen
1,Feb-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.42,US,Nielsen
2,March 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.6,Global,Nielsen
3,Apr-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.8M,US,Nielsen
4,May-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.28,US,Nielsen
5,June 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.24,US,Nielsen
6,July 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.69,US,Nielsen
7,Aug-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.43,US,Nielsen



--- Data Quality Issues ---
Duplicate rows: 25

Unique date formats (sample): ['Jan-2010' 'Feb-2010' 'March 2010' 'Apr-2010' 'May-2010' 'June 2010'
 'July 2010' 'Aug-2010' 'Sep-2010' 'Oct-2010']

media_type unique values: ['Broadcast TV' 'Cable TV' 'cable tv' 'CABLE TV' 'Print' 'print' 'Radio']

metric_value stored as string (sample):
       platform_name   report_month metric_value
3   CBS Evening News       Apr-2010        13.8M
15  CBS Evening News       Apr-2011        12.8M
59  CBS Evening News  December 2014        11.6M
82  CBS Evening News        2016/11        11.0M
85  CBS Evening News       Feb-2017        10.4M


### 1.6 – user_engagement_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT  
**Known issues:** Mixed date formats (`2015-01` vs `01/2015`), `retention_rate_pct` stored as decimal in some rows and `"78%"` string in others, nulls for new platforms in early months, 40 intentional duplicate rows

In [370]:
UserEngagement_df = pd.read_csv('Data Sources copy/user_engagement_monthly.csv')

print('Shape:', UserEngagement_df.shape)
print('Columns:', list(UserEngagement_df.columns))
print('\nData types:')
print(UserEngagement_df.dtypes)
print('\nNull counts:')
print(UserEngagement_df.isnull().sum())
print('\nSample rows:')
display(UserEngagement_df.head(6))

print('\n--- Data Quality Issues ---')
print('Duplicate rows:', UserEngagement_df.duplicated().sum())
print('\nMixed date formats (sample):', UserEngagement_df['year_month'].unique()[:10])
print('\nretention_rate_pct mixed types (sample):')
print(UserEngagement_df['retention_rate_pct'].head(20).tolist())

Shape: (4120, 8)
Columns: ['year_month', 'platform_name', 'media_type', 'avg_hours_per_user_per_month', 'monthly_active_users_millions', 'avg_sessions_per_user_per_month', 'retention_rate_pct', 'region']

Data types:
year_month                          object
platform_name                       object
media_type                          object
avg_hours_per_user_per_month       float64
monthly_active_users_millions      float64
avg_sessions_per_user_per_month    float64
retention_rate_pct                  object
region                              object
dtype: object

Null counts:
year_month                         0
platform_name                      0
media_type                         0
avg_hours_per_user_per_month       0
monthly_active_users_millions      0
avg_sessions_per_user_per_month    0
retention_rate_pct                 0
region                             0
dtype: int64

Sample rows:


,year_month,platform_name,media_type,avg_hours_per_user_per_month,monthly_active_users_millions,avg_sessions_per_user_per_month,retention_rate_pct,region
0,01/2015,Netflix,Streaming,32.75,64.72,9.1,0.9165,North America
1,2015-01,Netflix,Streaming,32.75,64.72,9.1,0.9165,Europe
2,2015-01,Netflix,Streaming,32.75,64.72,9.1,0.9165,APAC
3,03/2015,Netflix,Streaming,31.90,65.18,8.8,0.9268,North America
4,2015-03,Netflix,Streaming,31.90,65.18,8.8,92.7%,Europe
5,2015-03,Netflix,Streaming,31.90,65.18,8.8,0.9268,APAC



--- Data Quality Issues ---
Duplicate rows: 40

Mixed date formats (sample): ['01/2015' '2015-01' '03/2015' '2015-03' '2015-04' '2015-06' '06/2015'
 '2015-07' '2015-08' '09/2015']

retention_rate_pct mixed types (sample):
['0.9165', '0.9165', '0.9165', '0.9268', '92.7%', '0.9268', '0.9069', '90.7%', '0.9069', '0.8953', '89.5%', '0.8953', '0.9039', '0.9039', '0.9039', '0.9142', '0.9142', '0.9142', '0.9025', '90.2%']


### 1.7 – switch_factor_survey.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT, DIM_SWITCH_REASON  
**Known issues:** Inconsistent `switched_from` values (`Cable` vs `Cable TV` vs `cable tv`), many null `secondary_switch_reason`, duplicate `respondent_id` across survey years (re-contact design), mixed `satisfaction_score` types

In [373]:
SwitchSurvey_df = pd.read_csv('Data Sources copy/switch_factor_survey.csv')

print('Shape:', SwitchSurvey_df.shape)
print('Columns:', list(SwitchSurvey_df.columns))
print('\nData types:')
print(SwitchSurvey_df.dtypes)
print('\nNull counts:')
print(SwitchSurvey_df.isnull().sum())
print('\nSample rows:')
display(SwitchSurvey_df.head(5))

print('\n--- Data Quality Issues ---')
print('switched_from unique values:', SwitchSurvey_df['switched_from'].unique())
print('\nDuplicate respondent_ids (re-contact across years):')
print(SwitchSurvey_df[SwitchSurvey_df.duplicated('respondent_id', keep=False)][['survey_year','respondent_id']].head(6).to_string())
print('\nRows per survey year:')
print(SwitchSurvey_df.groupby('survey_year').size())

Shape: (2025, 10)
Columns: ['survey_year', 'respondent_id', 'switched_from', 'switched_to', 'primary_switch_reason', 'secondary_switch_reason', 'age_group', 'region', 'household_income_bracket', 'satisfaction_score']

Data types:
survey_year                   int64
respondent_id                 int64
switched_from                object
switched_to                  object
primary_switch_reason        object
secondary_switch_reason      object
age_group                    object
region                       object
household_income_bracket     object
satisfaction_score          float64
dtype: object

Null counts:
survey_year                    0
respondent_id                  0
switched_from                  0
switched_to                    0
primary_switch_reason          0
secondary_switch_reason     1105
age_group                      0
region                         0
household_income_bracket       0
satisfaction_score           173
dtype: int64

Sample rows:


,survey_year,respondent_id,switched_from,switched_to,primary_switch_reason,secondary_switch_reason,age_group,region,household_income_bracket,satisfaction_score
0,2015,10228,Broadcast TV,YouTube,Convenience,NaN,18-24,Midwest,$75K-$150K,6.89
1,2015,10051,Broadcast TV,Spotify,Content Selection,Recommendation,55+,South,$75K-$150K,8.00
2,2015,11518,newspaper,Amazon Prime Video,Device Availability,Device Availability,35-44,West,>$150K,8.00
3,2015,10563,Cable,Netflix,Price,NaN,18-24,South,<$35K,8.00
4,2015,10501,Cable,Max,Device Availability,NaN,55+,South,>$150K,6.89



--- Data Quality Issues ---
switched_from unique values: ['Broadcast TV' 'newspaper' 'Cable' 'broadcast tv' 'Satellite radio'
 'Satellite Radio' 'Cable television' 'Broadcast' 'Cable TV' 'cable tv'
 'Network TV' 'Newspaper' 'Print' 'satellite radio' 'CABLE TV'
 'Print Media' 'print media' 'SiriusXM']

Duplicate respondent_ids (re-contact across years):
   survey_year  respondent_id
1         2015          10051
2         2015          11518
3         2015          10563
4         2015          10501
5         2015          10457
6         2015          10285

Rows per survey year:
survey_year
2015    220
2016    200
2017    216
2018    200
2019    207
2020    199
2021    194
2022    185
2023    203
2024    201
dtype: int64


### 1.8 – platform_subscriber_monthly.csv
**Source:** Synthetic (AI-generated, anchored to real quarterly figures) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Null `revenue_usd_millions` for some platforms; seasonal churn spikes built in

In [376]:
PlatformSubscriber_df = pd.read_csv('Data Sources copy/platform_subscriber_monthly.csv')

print('Shape:', PlatformSubscriber_df.shape)
print('Columns:', list(PlatformSubscriber_df.columns))
print('\nData types:')
print(PlatformSubscriber_df.dtypes)
print('\nNull counts:')
print(PlatformSubscriber_df.isnull().sum())
print('\nSample rows:')
display(PlatformSubscriber_df.head(5))
print('\nPlatforms covered:', PlatformSubscriber_df['platform_name'].unique())
print('Date range:', PlatformSubscriber_df['year_month'].min(), '→', PlatformSubscriber_df['year_month'].max())

Shape: (640, 8)
Columns: ['year_month', 'platform_name', 'subscribers_millions', 'revenue_usd_millions', 'churn_rate_pct', 'new_subscribers_millions', 'cancelled_subscribers_millions', 'country_region']

Data types:
year_month                         object
platform_name                      object
subscribers_millions              float64
revenue_usd_millions              float64
churn_rate_pct                    float64
new_subscribers_millions          float64
cancelled_subscribers_millions    float64
country_region                     object
dtype: object

Null counts:
year_month                          0
platform_name                       0
subscribers_millions                0
revenue_usd_millions              344
churn_rate_pct                      0
new_subscribers_millions            0
cancelled_subscribers_millions      0
country_region                      0
dtype: int64

Sample rows:


,year_month,platform_name,subscribers_millions,revenue_usd_millions,churn_rate_pct,new_subscribers_millions,cancelled_subscribers_millions,country_region
0,2015-01,Netflix,65.32,729.9,0.0596,0.73,0.32,US
1,2015-02,Netflix,64.85,719.2,0.0393,0.74,0.21,US
2,2015-03,Netflix,64.69,701.7,0.0416,0.48,0.22,US
3,2015-04,Netflix,65.16,700.4,0.0343,0.29,0.19,US
4,2015-05,Netflix,64.34,688.3,0.0409,0.42,0.22,US



Platforms covered: ['Netflix' 'Hulu' 'Amazon Prime Video' 'Disney+' 'Max' 'Apple TV+'
 'Peacock' 'Paramount+']
Date range: 2015-01 → 2024-12


---
## Part 2 – Transformations: streaming_service.csv
**Target:** DIM_SUBSCRIPTION_PLAN (SCD3) and FACT_SUBSCRIPTION_PRICING

#### T1 – Remove Duplicates

In [380]:
StreamingServiceDistinct_df = StreamingService_df.drop_duplicates()

print('Rows before:', StreamingService_df.shape[0])
print('Rows after dedup:', StreamingServiceDistinct_df.shape[0])
print('Records removed:', StreamingService_df.shape[0] - StreamingServiceDistinct_df.shape[0])

print('\nNull counts after dedup:')
print(StreamingServiceDistinct_df.isnull().sum())

Rows before: 777
Rows after dedup: 777
Records removed: 0

Null counts after dedup:
service    0
date       0
price      0
dtype: int64


#### T2 – Date Standardization
Parse `date` from `%b-%Y` string format (e.g. `Jul-2011`) to proper `datetime`

In [383]:
print('Before:')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

StreamingServiceDistinct_df['date'] = pd.to_datetime(
    StreamingServiceDistinct_df['date'], format='%b-%Y'
)

print('\nAfter:')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

Before:
   service      date  price
0  Netflix  Jul-2011   7.99
1  Netflix  Aug-2011   7.99
2  Netflix  Sep-2011   7.99
3  Netflix  Oct-2011   7.99
4  Netflix  Nov-2011   7.99

After:
   service       date  price
0  Netflix 2011-07-01   7.99
1  Netflix 2011-08-01   7.99
2  Netflix 2011-09-01   7.99
3  Netflix 2011-10-01   7.99
4  Netflix 2011-11-01   7.99


#### T3 – Price Tier Classification
Derive `price_tier`: Budget ($0–$7) | Mid ($7–$14) | Premium (>$14)

In [386]:
print('Before — price distribution:')
print(StreamingServiceDistinct_df['price'].describe())

bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0-$7)', 'Mid ($7-$14)', 'Premium (>$14)']
StreamingServiceDistinct_df['price_tier'] = pd.cut(
    StreamingServiceDistinct_df['price'], bins=bins, labels=labels, right=True
)

print('\nAfter — sample rows:')
print(StreamingServiceDistinct_df[['service', 'date', 'price', 'price_tier']].head(10).to_string())
print('\nTier distribution:')
print(StreamingServiceDistinct_df['price_tier'].value_counts())

Before — price distribution:
count    777.000000
mean       9.378674
std        2.609859
min        4.990000
25%        7.990000
50%        8.990000
75%        9.990000
max       17.990000
Name: price, dtype: float64

After — sample rows:
   service       date  price    price_tier
0  Netflix 2011-07-01   7.99  Mid ($7-$14)
1  Netflix 2011-08-01   7.99  Mid ($7-$14)
2  Netflix 2011-09-01   7.99  Mid ($7-$14)
3  Netflix 2011-10-01   7.99  Mid ($7-$14)
4  Netflix 2011-11-01   7.99  Mid ($7-$14)
5  Netflix 2011-12-01   7.99  Mid ($7-$14)
6  Netflix 2012-01-01   7.99  Mid ($7-$14)
7  Netflix 2012-02-01   7.99  Mid ($7-$14)
8  Netflix 2012-03-01   7.99  Mid ($7-$14)
9  Netflix 2012-04-01   7.99  Mid ($7-$14)

Tier distribution:
price_tier
Mid ($7-$14)      560
Budget ($0-$7)    154
Premium (>$14)     63
Name: count, dtype: int64


#### T4 – Month-over-Month Price Change
Compute `price_change_mom` = diff of `price` per service sorted by date

In [389]:
StreamingServiceDistinct_df = StreamingServiceDistinct_df.sort_values(['service', 'date'])

StreamingServiceDistinct_df['price_change_mom'] = (
    StreamingServiceDistinct_df.groupby('service')['price'].diff()
)

print('Months where price changed:')
price_changes = (
    StreamingServiceDistinct_df[
        (StreamingServiceDistinct_df['price_change_mom'] != 0) &
        StreamingServiceDistinct_df['price_change_mom'].notna()
    ]
)
print(price_changes[['service', 'date', 'price', 'price_change_mom']].to_string())

Months where price changed:
         service       date  price  price_change_mom
588    Apple TV+ 2022-10-01   6.99               2.0
603    Apple TV+ 2024-01-01   9.99               3.0
167      Disney+ 2021-03-01   7.99               1.0
188      Disney+ 2022-12-01  10.99               3.0
198      Disney+ 2023-10-01  13.99               3.0
235      HBO Max 2023-02-01  15.99               1.0
319         Hulu 2021-10-01  12.99               1.0
333         Hulu 2022-12-01  14.99               2.0
346         Hulu 2024-01-01  17.99               3.0
90       Netflix 2019-01-01   8.99               1.0
126      Netflix 2022-01-01   9.99               1.0
147      Netflix 2023-10-01  15.49               5.5
451   Paramount+ 2023-06-01  11.99               2.0
641      Peacock 2023-08-01  11.99               2.0
552  Prime Video 2024-01-01  11.99               3.0
729      Shudder 2023-08-01   6.99               1.0


#### T5 – Cumulative Price Increase
Compute `cumulative_price_increase` = current price minus the service's first recorded price

In [392]:
StreamingServiceDistinct_df['cumulative_price_increase'] = (
    StreamingServiceDistinct_df.groupby('service')['price']
    .transform(lambda x: x - x.iloc[0])
)

print('Cumulative price increase per service (latest month):')
latest = (
    StreamingServiceDistinct_df
    .sort_values('date')
    .groupby('service')
    .last()
    .reset_index()
)
print(latest[['service', 'date', 'price', 'cumulative_price_increase']].to_string())

Cumulative price increase per service (latest month):
       service       date  price  cumulative_price_increase
0    Apple TV+ 2024-01-01   9.99                        5.0
1  Crunchyroll 2024-01-01   7.99                        0.0
2      Disney+ 2024-01-01  13.99                        7.0
3      HBO Max 2024-01-01  15.99                        1.0
4         Hulu 2024-01-01  17.99                        6.0
5      Netflix 2024-01-01  15.49                        7.5
6   Paramount+ 2024-01-01  11.99                        2.0
7      Peacock 2024-01-01  11.99                        2.0
8  Prime Video 2024-01-01  11.99                        3.0
9      Shudder 2024-01-01   6.99                        1.0


#### T6 – Service Name Normalization
Map legacy service names to canonical platform names for cross-source joins

In [395]:
name_map = {
    'HBO Max'     : 'Max',
    'Prime Video' : 'Amazon Prime Video'
}
StreamingServiceDistinct_df['platform_name'] = StreamingServiceDistinct_df['service'].replace(name_map)

print('Service → platform_name mapping:')
print(StreamingServiceDistinct_df[['service', 'platform_name']].drop_duplicates().to_string())

Service → platform_name mapping:
         service       platform_name
553    Apple TV+           Apple TV+
735  Crunchyroll         Crunchyroll
151      Disney+             Disney+
202      HBO Max                 Max
247         Hulu                Hulu
0        Netflix             Netflix
347   Paramount+          Paramount+
604      Peacock             Peacock
459  Prime Video  Amazon Prime Video
647      Shudder             Shudder


---
## Part 3 – Transformations: platform_summary.csv
**Target:** DIM_PLATFORM

#### T7 – Select Dimension Columns + Remove Duplicates

In [399]:
dim_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
PlatformSummaryDistinct_df = PlatformSummary_df[dim_cols].drop_duplicates()

print('Rows before:', PlatformSummary_df.shape[0])
print('Rows after dedup:', PlatformSummaryDistinct_df.shape[0])
print('\nNull counts (dim cols):')
print(PlatformSummaryDistinct_df.isnull().sum())
print('\nAll rows:')
display(PlatformSummaryDistinct_df)

Rows before: 12
Rows after dedup: 12

Null counts (dim cols):
platform          0
parent_company    0
content_focus     0
business_model    0
launch_year       0
dtype: int64

All rows:


,platform,parent_company,content_focus,business_model,launch_year
0,Netflix,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2007
1,Disney+,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2019
2,Hulu,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,2008
3,Amazon Prime Video,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2006
4,Paramount+,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2021
5,Peacock,Comcast,NBCUniversal content,Subscription + Ads,2020
6,Apple TV+,Apple Inc.,"Original series, films",Subscription,2019
7,Max,Warner Bros. Discovery,"HBO, Discovery, Warner Bros.",Subscription + Ads,2020
8,Roku,Roku Inc.,"FAST platform, advertising",AVOD + Transactional,2008
9,YouTube TV,Google,"User-generated, sports",AVOD + Subscription,2017


#### T8 – Derived Columns for DIM_PLATFORM
Add `is_digital`, `media_sector`, `platform_age_years`, `launch_decade`

In [402]:
print('Before:')
display(PlatformSummaryDistinct_df[['platform', 'launch_year']])

PlatformSummaryDistinct_df = PlatformSummaryDistinct_df.copy()
PlatformSummaryDistinct_df['is_digital']         = True
PlatformSummaryDistinct_df['media_sector']        = 'Streaming'
PlatformSummaryDistinct_df['platform_age_years']  = 2026 - PlatformSummaryDistinct_df['launch_year'].astype(int)
PlatformSummaryDistinct_df['launch_decade']       = (
    (PlatformSummaryDistinct_df['launch_year'].astype(int) // 10 * 10).astype(str) + 's'
)

print('\nAfter:')
display(PlatformSummaryDistinct_df[['platform', 'launch_year', 'launch_decade', 'is_digital', 'media_sector', 'platform_age_years']])

Before:


,platform,launch_year
0,Netflix,2007
1,Disney+,2019
2,Hulu,2008
3,Amazon Prime Video,2006
4,Paramount+,2021
5,Peacock,2020
6,Apple TV+,2019
7,Max,2020
8,Roku,2008
9,YouTube TV,2017



After:


,platform,launch_year,launch_decade,is_digital,media_sector,platform_age_years
0,Netflix,2007,2000s,True,Streaming,19
1,Disney+,2019,2010s,True,Streaming,7
2,Hulu,2008,2000s,True,Streaming,18
3,Amazon Prime Video,2006,2000s,True,Streaming,20
4,Paramount+,2021,2020s,True,Streaming,5
5,Peacock,2020,2020s,True,Streaming,6
6,Apple TV+,2019,2010s,True,Streaming,7
7,Max,2020,2020s,True,Streaming,6
8,Roku,2008,2000s,True,Streaming,18
9,YouTube TV,2017,2010s,True,Streaming,9


#### T9 – Cross-Source Platform Alignment
Identify platforms in pricing data that are missing from platform_summary (need manual enrichment)

In [405]:
services_in_pricing  = set(StreamingServiceDistinct_df['platform_name'].unique())
platforms_in_summary = set(PlatformSummaryDistinct_df['platform'].unique())

print('In pricing but NOT in platform_summary (will add manually):')
print(sorted(services_in_pricing - platforms_in_summary))

print('\nIn platform_summary but NOT in pricing (no pricing history):')
print(sorted(platforms_in_summary - services_in_pricing))

In pricing but NOT in platform_summary (will add manually):
['Crunchyroll', 'Shudder']

In platform_summary but NOT in pricing (no pricing history):
['Pluto TV', 'Roku', 'Tubi', 'YouTube TV']


---
## Part 4 – Build Dimensions: DIM_SUBSCRIPTION_PLAN (SCD2) & DIM_PLATFORM (SCD3)

**Design rationale:**
- `DIM_SUBSCRIPTION_PLAN` → **SCD2**: prices change frequently (Netflix alone: $7.99→$9.99→$13.99→$15.49→$17.99). Full price history needed to tie revenue and churn analysis to the price active at any point in time.
- `DIM_PLATFORM` → **SCD3**: parent company and rebrand changes are rare. Only need current vs. previous state in the same row.

### 4.1 – DIM_SUBSCRIPTION_PLAN (SCD2)

Collapse consecutive months at the same price per platform into one SCD2 record.  
Each price change gets a **new row** with its own surrogate key, `effective_date`, `end_date`, and `is_current`.

In [409]:
# Group by platform + price ONLY — price_tier is categorical and causes
# a cross-product with all tier labels if included in groupby
pricing = StreamingServiceDistinct_df[['platform_name', 'date', 'price']].copy()
pricing = pricing.sort_values(['platform_name', 'date'])

# Detect price changes
pricing['price_changed'] = pricing.groupby('platform_name')['price'].diff().ne(0)
pricing['price_group']   = pricing.groupby('platform_name')['price_changed'].cumsum()

#Collapsing each consecutive same-price block into one SCD2 record
DimSubscriptionPlan = (
    pricing
    .groupby(['platform_name', 'price_group', 'price'])
    .agg(effective_date=('date', 'min'), end_date=('date', 'max'))
    .reset_index()
    .drop(columns='price_group')
)

#Deriving price_tier AFTER collapsing 
bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0-$7)', 'Mid ($7-$14)', 'Premium (>$14)']
DimSubscriptionPlan['price_tier'] = pd.cut(
    DimSubscriptionPlan['price'], bins=bins, labels=labels, right=True
).astype(str)

#Mark is_current = latest record per platform has no end date
latest_idx = DimSubscriptionPlan.groupby('platform_name')['end_date'].idxmax()
DimSubscriptionPlan['is_current'] = False
DimSubscriptionPlan.loc[latest_idx, 'is_current'] = True
DimSubscriptionPlan.loc[DimSubscriptionPlan['is_current'], 'end_date'] = None

#Surrogate key
DimSubscriptionPlan['plan_key'] = range(1, len(DimSubscriptionPlan) + 1)

DimSubscriptionPlan = DimSubscriptionPlan[[
    'plan_key', 'platform_name', 'price', 'price_tier',
    'effective_date', 'end_date', 'is_current'
]]

n_total   = len(DimSubscriptionPlan)
n_current = DimSubscriptionPlan['is_current'].sum()
print(f'DimSubscriptionPlan: {n_total} rows ({n_current} current, {n_total - n_current} historical)')
print('\nNetflix price history (all SCD2 rows):')
display(DimSubscriptionPlan[DimSubscriptionPlan['platform_name'] == 'Netflix'])

DimSubscriptionPlan: 26 rows (10 current, 16 historical)

Netflix price history (all SCD2 rows):


,plan_key,platform_name,price,price_tier,effective_date,end_date,is_current
16,17,Netflix,7.99,Mid ($7-$14),2011-07-01,2018-12-01,False
17,18,Netflix,8.99,Mid ($7-$14),2019-01-01,2021-12-01,False
18,19,Netflix,9.99,Mid ($7-$14),2022-01-01,2023-09-01,False
19,20,Netflix,15.49,Premium (>$14),2023-10-01,NaT,True


#### SCD2 Delta Demo — Netflix price increase ($15.49 → $17.99, Jan 2024)

Simulates an incoming price change. Expire the current row, insert a new row with a new surrogate key.

In [412]:
delta_date    = pd.Timestamp('2024-01-01')
new_price     = 17.99
new_tier      = 'Premium (>$14)'
platform_name = 'Netflix'

# Expiring the current row
mask = (DimSubscriptionPlan['platform_name'] == platform_name) & (DimSubscriptionPlan['is_current'] == True)
DimSubscriptionPlan.loc[mask, 'end_date']   = delta_date - pd.Timedelta(days=1)
DimSubscriptionPlan.loc[mask, 'is_current'] = False

# Inserting new row with new surrogate key
new_row = {
    'plan_key'      : int(DimSubscriptionPlan['plan_key'].max()) + 1,
    'platform_name' : platform_name,
    'price'         : new_price,
    'price_tier'    : new_tier,
    'effective_date': delta_date,
    'end_date'      : None,
    'is_current'    : True
}
DimSubscriptionPlan = pd.concat(
    [DimSubscriptionPlan, pd.DataFrame([new_row])], ignore_index=True
)

print('\n=== AFTER — full Netflix price history preserved (new row inserted) ===')
display(DimSubscriptionPlan[DimSubscriptionPlan['platform_name'] == platform_name])


=== AFTER — full Netflix price history preserved (new row inserted) ===


,plan_key,platform_name,price,price_tier,effective_date,end_date,is_current
16,17,Netflix,7.99,Mid ($7-$14),2011-07-01,2018-12-01,False
17,18,Netflix,8.99,Mid ($7-$14),2019-01-01,2021-12-01,False
18,19,Netflix,9.99,Mid ($7-$14),2022-01-01,2023-09-01,False
19,20,Netflix,15.49,Premium (>$14),2023-10-01,2023-12-31,False
26,27,Netflix,17.99,Premium (>$14),2024-01-01,NaT,True


### 4.2 – DIM_PLATFORM (SCD3)

Parent company and rebrand changes are rare — only current vs. previous needed in the same row.  
SCD3-tracked: `current_business_model` / `previous_business_model`, `current_parent_company` / `previous_parent_company`

In [415]:
# Platforms in pricing data not covered by platform_summary — built from metadata
NewPlatformRows_df = pd.DataFrame([
    {
        'platform': 'Crunchyroll', 'parent_company': 'Sony Group Corporation',
        'content_focus': 'Anime, manga', 'business_model': 'Subscription',
        'launch_year': 2009, 'is_digital': True, 'media_sector': 'Streaming',
        'platform_age_years': 2026 - 2009, 'launch_decade': '2000s'
    },
    {
        'platform': 'Shudder', 'parent_company': 'AMC Networks',
        'content_focus': 'Horror, thriller', 'business_model': 'Subscription',
        'launch_year': 2015, 'is_digital': True, 'media_sector': 'Streaming',
        'platform_age_years': 2026 - 2015, 'launch_decade': '2010s'
    },
])

DimPlatform = pd.concat([PlatformSummaryDistinct_df, NewPlatformRows_df], ignore_index=True)

# Rename 'platform' to 'platform_name' for consistency
if 'platform' in DimPlatform.columns:
    DimPlatform = DimPlatform.rename(columns={'platform': 'platform_name'})

# SCD3 columns — initial load: previous_ columns are NULL
DimPlatform['current_parent_company']    = DimPlatform['parent_company']
DimPlatform['previous_parent_company']   = None
DimPlatform['current_business_model']    = DimPlatform['business_model']
DimPlatform['previous_business_model']   = None
DimPlatform['parent_company_change_date']= None

# Surrogate key
DimPlatform['platform_key'] = range(1, len(DimPlatform) + 1)

display_cols = [
    'platform_key', 'platform_name',
    'current_parent_company', 'previous_parent_company',
    'current_business_model', 'previous_business_model',
    'parent_company_change_date', 'is_digital', 'launch_year'
]

print('DimPlatform initial load (previous_ columns all NULL):')
display(DimPlatform[display_cols])

DimPlatform initial load (previous_ columns all NULL):


,platform_key,platform_name,current_parent_company,previous_parent_company,current_business_model,previous_business_model,parent_company_change_date,is_digital,launch_year
0,1,Netflix,Netflix Inc.,None,Subscription + Ads,None,None,True,2007
1,2,Disney+,The Walt Disney Company,None,Subscription + Ads,None,None,True,2019
2,3,Hulu,The Walt Disney Company,None,Subscription + Ads,None,None,True,2008
3,4,Amazon Prime Video,Amazon.com Inc.,None,Prime bundle + Subscription,None,None,True,2006
4,5,Paramount+,Paramount Global,None,Subscription + Ads,None,None,True,2021
5,6,Peacock,Comcast,None,Subscription + Ads,None,None,True,2020
6,7,Apple TV+,Apple Inc.,None,Subscription,None,None,True,2019
7,8,Max,Warner Bros. Discovery,None,Subscription + Ads,None,None,True,2020
8,9,Roku,Roku Inc.,None,AVOD + Transactional,None,None,True,2008
9,10,YouTube TV,Google,None,AVOD + Subscription,None,None,True,2017


#### SCD3 Delta Demo — Max business model change (May 2023)

Simulates: Max changes business model from `Subscription` to `Subscription + Ads` after the HBO Max rebrand.  
Process: shift current → previous **in the same row**, write new current value. No new row inserted.

In [418]:
change_date    = pd.Timestamp('2023-05-23')
new_biz_model  = 'Subscription + Ads'
platform_delta = 'Max'

print('=== BEFORE SCD3 delta ===')
display(DimPlatform[DimPlatform['platform_name'] == platform_delta][display_cols])

# SCD3: shift current → previous, write new current value in the same row
mask = DimPlatform['platform_name'] == platform_delta
DimPlatform.loc[mask, 'previous_business_model']    = DimPlatform.loc[mask, 'current_business_model']
DimPlatform.loc[mask, 'current_business_model']     = new_biz_model
DimPlatform.loc[mask, 'parent_company_change_date'] = change_date

print('\n=== AFTER SCD3 delta — current + previous in same row, no new row created ===')
display(DimPlatform[DimPlatform['platform_name'] == platform_delta][display_cols])

=== BEFORE SCD3 delta ===


,platform_key,platform_name,current_parent_company,previous_parent_company,current_business_model,previous_business_model,parent_company_change_date,is_digital,launch_year
7,8,Max,Warner Bros. Discovery,None,Subscription + Ads,None,None,True,2020



=== AFTER SCD3 delta — current + previous in same row, no new row created ===


,platform_key,platform_name,current_parent_company,previous_parent_company,current_business_model,previous_business_model,parent_company_change_date,is_digital,launch_year
7,8,Max,Warner Bros. Discovery,None,Subscription + Ads,Subscription + Ads,2023-05-23 00:00:00,True,2020


---
## Part 5 – Connect to PostgreSQL (`cs689` / `media` schema)

`dim_subscription_plan` is already loaded. This section connects to the database  
and builds + loads `fact_subscription_pricing`.


In [421]:
import psycopg2
from psycopg2.extras import execute_values

# ── Connection — fill in your credentials ────────────────────────────────────
conn = psycopg2.connect(
    host     = 'localhost',
    database = 'cs689',
    user     = 'postgres',
    password = 'medha25'
)
cursor = conn.cursor()

# Create schema if it doesn't exist
cursor.execute('CREATE SCHEMA IF NOT EXISTS "media"')
conn.commit()

print("Connected to cs689 | schema media ready")

Connected to cs689 | schema media ready


> Table is now available in pgAdmin / psql as `"media".dim_subscription_plan`.  
> Run your BQ4 analytical query directly in the PostgreSQL query tool.

```sql
-- Close connection when done
-- cursor.close()
-- conn.close()
```

---
## Part 6 – Build & Load FACT_SUBSCRIPTION_PRICING

Joins the cleaned pricing data (streaming_service.csv) with subscriber/revenue data  
(platform_subscriber_monthly.csv) and links to `DIM_SUBSCRIPTION_PLAN` via `plan_key`.

**Grain:** Platform × Month  
**Measures:** `price`, `subscribers_millions`, `revenue_usd_millions`, `churn_rate_pct`  
**Foreign keys:** `plan_key` → DIM_SUBSCRIPTION_PLAN


### 6.1 – Load & preview platform_subscriber_monthly.csv

In [ ]:
Subscriber_df = pd.read_csv('Data Sources copy/platform_subscriber_monthly.csv')

print('Shape:', Subscriber_df.shape)
print('Columns:', list(Subscriber_df.columns))
print('\nNull counts:')
print(Subscriber_df.isnull().sum())
print('\nSample:')
display(Subscriber_df.head(6))

### 6.2 – Parse dates and normalize platform names

In [ ]:
# Parse year_month to datetime
Subscriber_df['date'] = pd.to_datetime(Subscriber_df['year_month'], format='%Y-%m')

# Only keep US/global rows (no country split needed for BQ4)
# platform_subscriber_monthly has one row per platform per month
Subscriber_df = Subscriber_df.drop_duplicates(subset=['platform_name', 'date'])

print('After dedup:', Subscriber_df.shape)
print('Platforms:', Subscriber_df['platform_name'].unique())

### 6.3 – Join pricing + subscriber data

In [ ]:
# StreamingServiceDistinct_df already has: platform_name, date, price, price_tier, price_change_mom
pricing_cols = StreamingServiceDistinct_df[[
    'platform_name', 'date', 'price', 'price_tier', 'price_change_mom'
]].copy()

subscriber_cols = Subscriber_df[[
    'platform_name', 'date',
    'subscribers_millions', 'revenue_usd_millions', 'churn_rate_pct'
]].copy()

# Left join: pricing drives the grain (every platform-month with a price)
FactSubscriptionPricing = pd.merge(
    pricing_cols, subscriber_cols,
    on=['platform_name', 'date'],
    how='left'
)

print('FactSubscriptionPricing shape:', FactSubscriptionPricing.shape)
print('\nNull check:')
print(FactSubscriptionPricing.isnull().sum())
display(FactSubscriptionPricing.head(8))

### 6.4 – Resolve `plan_key` foreign key from DIM_SUBSCRIPTION_PLAN

In [ ]:
# Vectorized plan_key lookup: merge on platform_name + price, then filter by date range
dim = DimSubscriptionPlan[['plan_key', 'platform_name', 'price', 'effective_date', 'end_date']].copy()
dim['effective_date'] = pd.to_datetime(dim['effective_date'])
dim['end_date_fill']  = pd.to_datetime(dim['end_date']).fillna(pd.Timestamp('2099-12-31'))

# Merge on platform + price, then keep rows where date is in the plan's validity window
merged = FactSubscriptionPricing.merge(
    dim[['plan_key', 'platform_name', 'price', 'effective_date', 'end_date_fill']],
    on=['platform_name', 'price'],
    how='left'
)
merged = merged[
    (merged['date'] >= merged['effective_date']) &
    (merged['date'] <= merged['end_date_fill'])
]
# Drop duplicate matches (shouldn't exist but safety net)
merged = merged.drop_duplicates(subset=['platform_name', 'date'])

FactSubscriptionPricing = FactSubscriptionPricing.merge(
    merged[['platform_name', 'date', 'plan_key']],
    on=['platform_name', 'date'],
    how='left'
)

print('plan_key null count:', FactSubscriptionPricing['plan_key'].isnull().sum())
print('Rows with matched plan_key:', FactSubscriptionPricing['plan_key'].notna().sum())
display(FactSubscriptionPricing[['platform_name','date','price','plan_key','subscribers_millions']].head(8))

### 6.5 – Add surrogate key and finalize

In [ ]:
FactSubscriptionPricing['fact_pricing_key'] = range(1, len(FactSubscriptionPricing) + 1)

# Final column order
FactSubscriptionPricing = FactSubscriptionPricing[[
    'fact_pricing_key', 'plan_key', 'platform_name', 'date',
    'price', 'price_tier', 'price_change_mom',
    'subscribers_millions', 'revenue_usd_millions', 'churn_rate_pct'
]]

print('Final shape:', FactSubscriptionPricing.shape)
print('\nSample — Netflix rows:')
display(FactSubscriptionPricing[FactSubscriptionPricing['platform_name'] == 'Netflix'].head(10))

### 6.6 – Load FACT_SUBSCRIPTION_PRICING into PostgreSQL

In [ ]:
cursor.execute("""
    DROP TABLE IF EXISTS "media".fact_subscription_pricing;
    CREATE TABLE "media".fact_subscription_pricing (
        fact_pricing_key         SERIAL PRIMARY KEY,
        plan_key                 INTEGER,
        platform_name            TEXT,
        date                     DATE,
        price                    NUMERIC(6,2),
        price_tier               TEXT,
        price_change_mom         NUMERIC(6,2),
        subscribers_millions     NUMERIC(10,3),
        revenue_usd_millions     NUMERIC(12,2),
        churn_rate_pct           NUMERIC(6,4)
    );
""")
conn.commit()

fact_load = FactSubscriptionPricing[[
    'plan_key', 'platform_name', 'date',
    'price', 'price_tier', 'price_change_mom',
    'subscribers_millions', 'revenue_usd_millions', 'churn_rate_pct'
]].copy()
fact_load['date']       = pd.to_datetime(fact_load['date']).dt.date
fact_load['price_tier'] = fact_load['price_tier'].astype(str)

rows = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in fact_load.itertuples(index=False)
]

execute_values(cursor, """
    INSERT INTO "media".fact_subscription_pricing
        (plan_key, platform_name, date, price, price_tier, price_change_mom,
         subscribers_millions, revenue_usd_millions, churn_rate_pct)
    VALUES %s
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.fact_subscription_pricing")

# Verify — Netflix sample
cursor.execute("""
    SELECT f.platform_name, f.date, f.price, f.price_tier,
           f.subscribers_millions, f.churn_rate_pct, d.is_current
    FROM "media".fact_subscription_pricing f
    JOIN "media".dim_subscription_plan d ON f.plan_key = d.plan_key
    WHERE f.platform_name = 'Netflix'
    ORDER BY f.date
    LIMIT 10
""")
rows_back = cursor.fetchall()
verify_fact = pd.DataFrame(rows_back, columns=[
    'platform_name','date','price','price_tier','subscribers_millions','churn_rate_pct','is_current'
])
display(verify_fact)

> `media.fact_subscription_pricing` loaded — 777 rows, one per platform × month.  
> Run BQ4 analytical query in pgAdmin joining this table with `media.dim_subscription_plan`.
